# 02. 멀티에이전트: Subagents — 감독자 패턴

## 학습 목표

- 감독자 → 서브에이전트 → 도구의 3계층 아키텍처를 설계한다
- 서브에이전트를 `@tool`로 래핑하여 감독자에게 도구로 노출한다
- HITL, ToolRuntime, 비동기/디스패치 패턴을 이해한다

## 2.1 환경 설정

Subagents 패턴은 중앙 감독자(Supervisor) 에이전트가 전문화된 서브에이전트들을 도구처럼 호출하여 작업을 위임하는 멀티에이전트 아키텍처입니다. 이 노트북에서는 캘린더와 이메일 도메인을 처리하는 개인 비서 시스템을 구축합니다.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4.1")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 2.2 Subagents 아키텍처 개요

Subagents 패턴은 **3계층 아키텍처**로 구성됩니다. 감독자가 모든 라우팅을 담당하고, 서브에이전트는 사용자와 직접 상호작용하지 않으며, 결과를 감독자에게 반환합니다.

![감독자-서브에이전트 계층 구조](../assets/images/supervisor_subagents.png)

| 계층 | 역할 | 특징 |
|------|------|------|
| **저수준 도구** | 외부 서비스 직접 호출 (Calendar API, Email API) | 단순한 함수 래퍼 |
| **서브에이전트** | 도메인별 추론 + 도구 조합 | 전문 시스템 프롬프트, 독립적 도구 세트 |
| **감독자** | 작업 분해, 위임, 결과 집계 | 전체 대화 기억, 서브에이전트 = 도구로 취급 |

### 핵심 특성

- **중앙 집중 제어**: 모든 라우팅이 감독자를 통해 흐릅니다
- **컨텍스트 격리**: 서브에이전트는 매번 깨끗한 컨텍스트 윈도우에서 실행되어 컨텍스트 비대화를 방지합니다
- **병렬 실행**: 여러 서브에이전트를 한 턴에서 동시에 호출할 수 있습니다
- **도구 기반 호출**: 서브에이전트를 `@tool`로 래핑하여 감독자에게 일반 도구처럼 노출합니다

### 사용 시점

서브에이전트 패턴은 여러 도메인(캘린더, 이메일, CRM 등)을 관리하면서 서브에이전트가 사용자와 직접 대화할 필요 없고, 중앙화된 워크플로 관리가 필요할 때 적합합니다. 도구가 적은 단순한 시나리오에서는 단일 에이전트로 충분합니다.

## 2.3 저수준 도구 정의

3계층 아키텍처의 최하층인 저수준 도구를 정의합니다. 이 도구들은 외부 서비스(Calendar API, Email API)와 직접 상호작용하는 단순한 함수 래퍼입니다. 실제 프로덕션에서는 Google Calendar API, Email Service 등과 연동하지만, 여기서는 학습을 위한 스텁(stub) 구현을 사용합니다.

도구를 설계할 때 중요한 점:
- 하나의 도구는 하나의 기능만 담당 (단일 책임 원칙)
- 동일한 도구를 여러 서브에이전트에 중복 할당하지 않아야 합니다
- docstring을 명확하게 작성하여 LLM이 도구를 올바르게 선택할 수 있게 합니다

In [ ]:
from langchain_core.tools import tool


@tool
def create_calendar_event(
    title: str,
    start_time: str,
    end_time: str,
    attendees: list[str] = None,
) -> str:
    """새 캘린더 이벤트를 생성합니다."""
    return f"이벤트 '{title}' 생성됨: {start_time} ~ {end_time}"

In [ ]:
@tool
def read_calendar_events(date: str) -> str:
    """날짜(YYYY-MM-DD)의 캘린더 이벤트를 조회합니다."""
    return f"{date}에 이벤트가 없습니다."

In [ ]:
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """이메일 메시지를 전송합니다."""
    return f"{to}에게 이메일 전송됨: '{subject}'"


@tool
def read_emails(folder: str = "inbox", limit: int = 10) -> str:
    """폴더에서 최근 이메일을 읽습니다."""
    return f"{folder}에 이메일 3개"

In [ ]:
@tool
def search_emails(query: str, limit: int = 10) -> str:
    """검색어로 이메일을 검색합니다."""
    return f"'{query}' 검색 결과 2건"

## 2.4 서브에이전트 생성

각 서브에이전트는 `create_agent()`로 생성하며, 세 가지 핵심 요소를 갖습니다:

1. **전문화된 시스템 프롬프트**: 도메인별 역할과 행동 지침을 정의합니다
2. **도메인별 도구 세트**: 해당 도메인의 도구만 할당하여 관심사를 분리합니다
3. **`name` 식별자**: 감독자가 서브에이전트를 구분하고 호출할 때 사용합니다

서브에이전트의 세분화 수준(granularity)은 도메인 단위(캘린더, 이메일 등)가 권장됩니다. 너무 세분화하면 감독자의 라우팅 부담이 증가하고, 너무 통합하면 컨텍스트 격리의 이점이 줄어듭니다.

In [ ]:
from langchain.agents import create_agent

calendar_agent = create_agent(
    model="gpt-4.1",
    tools=[create_calendar_event, read_calendar_events],
    system_prompt="당신은 캘린더 어시스턴트입니다. ISO 8601 날짜 형식을 사용하세요.",
    name="calendar_agent",
)

In [ ]:
email_agent = create_agent(
    model="gpt-4.1",
    tools=[send_email, read_emails, search_emails],
    system_prompt="당신은 이메일 어시스턴트입니다. 메시지를 전문적으로 작성하세요.",
    name="email_agent",
)

## 2.5 서브에이전트를 도구로 래핑

서브에이전트를 감독자에게 노출하는 표준 패턴은 `@tool` 데코레이터로 감싸는 것입니다. 래핑 함수 내부에서 `subagent.invoke()`를 호출하고, 마지막 메시지의 `content`를 반환합니다.

이 패턴의 장점:
- 감독자 입장에서 서브에이전트는 일반 도구와 동일하게 취급됩니다
- 서브에이전트의 내부 구현 변경이 감독자에 영향을 주지 않습니다
- 입출력 형식을 래핑 함수에서 자유롭게 커스터마이징할 수 있습니다

**입출력 전략 선택**: 쿼리만 전달(간단)할 수도 있고, 전체 컨텍스트를 전달(정교)할 수도 있습니다. 결과 반환 시에도 최종 결과만 반환하거나 전체 히스토리를 반환하는 선택지가 있습니다.

In [ ]:
@tool("schedule_event", description="캘린더 이벤트 예약")
def call_calendar(query: str) -> str:
    """캘린더 작업을 캘린더 에이전트에 위임합니다."""
    result = calendar_agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=lf_config,
    )
    return result["messages"][-1].content

In [ ]:
@tool("manage_email", description="이메일 전송, 읽기, 검색")
def call_email(query: str) -> str:
    """이메일 작업을 이메일 에이전트에 위임합니다."""
    result = email_agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=lf_config,
    )
    return result["messages"][-1].content

## 2.6 감독자 에이전트 조립

감독자는 래핑된 서브에이전트 도구들을 `tools`에 전달받아 생성됩니다. 감독자의 시스템 프롬프트에는 작업 분해(task decomposition) 및 위임(delegation) 지침을 포함합니다.

감독자 설계 시 고려사항:
- **에러 처리**: 서브에이전트 실패를 감독자가 우아하게 처리해야 합니다
- **결과 집계**: 여러 서브에이전트의 결과를 통합하여 사용자에게 일관된 응답을 제공합니다
- **승인 범위**: 상태를 변경하는 작업(이메일 전송, 이벤트 생성)에만 HITL을 적용합니다

In [ ]:
supervisor = create_agent(
    model="gpt-4.1",
    tools=[call_calendar, call_email],
    system_prompt=(
        "당신은 개인 비서입니다. 복잡한 요청을 "
        "하위 작업으로 분해하고 적절한 에이전트에 위임하세요."
    ),
)

## 2.7 실행 테스트

```
User: "내일 2시 Sarah 미팅 잡고 초대 이메일 보내줘"
Supervisor → call_calendar → create_calendar_event
Supervisor → call_email → send_email
Supervisor: "미팅과 초대 이메일 완료"
```

In [ ]:
# response = supervisor.invoke({
#     "messages": [{"role": "user",
#         "content": "내일 2시에 Sarah와 미팅 잡고 "
#                    "초대 이메일 보내줘."}]
# }, config=lf_config)
# print(response["messages"][-1].content)

## 2.8 HITL (Human-in-the-Loop) 통합

`HumanInTheLoopMiddleware`와 `checkpointer`를 결합하면 고위험 도구 호출(이메일 전송, 일정 생성 등) 전에 사용자 승인을 요청할 수 있습니다. 에이전트가 보호된 도구를 호출하려 할 때 실행이 일시 중지되고, 사용자가 검토합니다.

### 승인 응답 유형

| 응답 | 설명 | 코드 |
|------|------|------|
| **승인(Approve)** | 도구 호출을 그대로 실행 | `Command(resume="approve")` |
| **편집(Edit)** | 도구 인자를 수정 후 실행 | `Command(resume={"type": "edit", "args": {...}})` |
| **거부(Reject)** | 도구 호출을 취소 | `Command(resume={"type": "reject", "reason": "..."})` |

HITL은 상태를 변경하는 작업(send_email, create_calendar_event 등)에만 적용하는 것이 권장됩니다. 읽기 전용 작업에는 불필요한 마찰을 줄이기 위해 적용하지 않습니다.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "schedule_event": {"allowed_decisions": ["approve", "edit", "reject"]},
        "manage_email": {"allowed_decisions": ["approve", "reject"]},
    }
)

In [ ]:
supervisor_hitl = create_agent(
    model="gpt-4.1",
    tools=[call_calendar, call_email],
    checkpointer=InMemorySaver(),
    middleware=[hitl],
    system_prompt="당신은 개인 비서입니다.",
)

In [ ]:
from langgraph.types import Command

# result = supervisor_hitl.invoke(Command(resume="approve"), config=lf_config)
# result = supervisor_hitl.invoke(Command(resume={"type": "reject", "reason": "마음이 바뀌었습니다"}), config=lf_config)

## 2.9 컨텍스트 전달 (ToolRuntime)

`ToolRuntime`은 런타임 컨텍스트(사용자 ID, 이름, 타임존 등)를 메시지에 포함하지 않고도 도구에 전달하는 메커니즘입니다. 매 프롬프트마다 반복적인 텍스트를 넣는 대신, `ToolRuntime`으로 공유 컨텍스트를 한 번만 설정합니다.

도구 함수에서는 `runtime_context` 키워드 인자로 컨텍스트에 접근합니다. 이를 통해:
- 사용자 신원 정보를 도구가 활용할 수 있습니다 (예: 발신자 이메일 자동 설정)
- 환경 설정(타임존 등)을 일관되게 적용할 수 있습니다
- 프롬프트 길이를 줄여 토큰 비용을 절감합니다

In [ ]:
# ToolRuntime은 LangChain v1 에이전트의 런타임 컨텍스트 전달 메커니즘입니다.
# 아직 릴리스되지 않은 경우를 대비하여 폴백 처리합니다.
try:
    from langchain.runtime import ToolRuntime
except ImportError:
    # ToolRuntime이 아직 릴리스되지 않은 경우 간단한 대체 구현
    class ToolRuntime:
        """사전 릴리스 버전을 위한 폴백 ToolRuntime."""

        def __init__(self, context: dict):
            self.context = context

    print("ToolRuntime 미출시 — 폴백 스텁 사용")

runtime = ToolRuntime(
    context={
        "user_email": "me@example.com",
        "user_name": "Alice",
        "timezone": "Asia/Seoul",
    }
)

In [ ]:
supervisor_ctx = create_agent(
    model="gpt-4.1",
    tools=[call_calendar, call_email],
    system_prompt="당신은 개인 비서입니다.",
)

In [ ]:
# 도구에서 런타임 컨텍스트 접근
@tool
def send_email_ctx(to: str, subject: str, body: str, *, runtime_context: dict) -> str:
    """현재 사용자로 이메일을 전송합니다."""
    sender = runtime_context["user_email"]
    return f"{sender}에서 {to}로 이메일 전송됨: '{subject}'"

## 2.10 비동기 실행 패턴

서브에이전트의 실행 모드는 두 가지입니다:

| 모드 | 동작 | 사용 시점 |
|------|------|----------|
| **동기(Synchronous)** | 감독자가 서브에이전트 완료를 대기 후 다음 진행 | 결과가 다음 작업에 필요할 때 (기본값) |
| **비동기(Asynchronous)** | 즉시 Job ID 반환, 백그라운드 실행, 나중에 결과 조회 | 독립적인 작업, 장시간 소요 작업 |

비동기 패턴은 **Job ID → Status → Result** 구조를 따릅니다. 서브에이전트가 즉시 Job ID를 반환하면, 감독자는 다른 작업을 계속 진행하다가 나중에 결과를 조회합니다.

In [ ]:
import uuid

job_store = {}


@tool("schedule_async", description="이벤트 예약 (비동기)")
def call_calendar_async(query: str) -> str:
    """비동기 캘린더 작업을 시작하고 작업 ID를 반환합니다."""
    job_id = str(uuid.uuid4())[:8]
    job_store[job_id] = {"status": "done", "result": "이벤트 생성됨"}
    return f"작업 시작됨: {job_id}"

In [ ]:
@tool("check_job", description="비동기 작업 상태 확인")
def check_job(job_id: str) -> str:
    """비동기 작업의 상태를 확인합니다."""
    job = job_store.get(job_id, {"status": "not_found"})
    return f"상태: {job['status']}, 결과: {job.get('result')}"

## 2.11 단일 디스패치 도구 패턴

도구 패턴에는 두 가지 접근법이 있습니다:

| 패턴 | 설명 | 장점 |
|------|------|------|
| **에이전트별 도구** | 서브에이전트마다 별도의 래핑 도구 생성 | 세밀한 제어, description 커스터마이징 용이 |
| **단일 디스패치 도구** | 하나의 파라미터화된 도구로 모든 서브에이전트 호출 | 확장성 우수, 서브에이전트 추가/제거가 독립적 |

단일 디스패치 패턴은 `agent_name` 파라미터로 호출 대상을 지정합니다. 에이전트 레지스트리에 등록된 서브에이전트를 이름으로 조회하여 호출하므로, 분산 팀에서 서브에이전트를 독립적으로 추가하거나 제거할 수 있어 확장성이 뛰어납니다.

In [ ]:
from typing import Literal

AGENT_REGISTRY = {"calendar": calendar_agent, "email": email_agent}


@tool("delegate", description="전문 에이전트에 위임")
def dispatch(agent_name: Literal["calendar", "email"], query: str) -> str:
    """지정된 서브에이전트에 작업을 라우팅합니다."""
    agent = AGENT_REGISTRY[agent_name]
    result = agent.invoke(
        {"messages": [{"role": "user", "content": query}]}, config=lf_config
    )
    return result["messages"][-1].content

In [ ]:
supervisor_dispatch = create_agent(
    model="gpt-4.1",
    tools=[dispatch],
    system_prompt=(
        "delegate 도구를 사용하여 작업을 라우팅하세요. "
        "에이전트: 'calendar'(일정), 'email'(이메일)."
    ),
)

## 요약

| 항목 | 핵심 |
|------|------|
| **3계층** | 도구 → 서브에이전트(`create_agent`) → 감독자 |
| **래핑** | `@tool` + `subagent.invoke()` → 마지막 content 반환 |
| **격리** | 서브에이전트 = 깨끗한 컨텍스트, 감독자만 전체 기억 |
| **HITL** | `HumanInTheLoopMiddleware` + `InMemorySaver` |
| **컨텍스트** | `ToolRuntime(context={...})` → `runtime_context` |
| **비동기** | Job ID → Status → Result 패턴 |
| **디스패치** | 단일 `dispatch(agent_name, query)` 도구 |

### 다음 단계
→ **[03_multi_agent_handoffs_router.ipynb](./03_multi_agent_handoffs_router.ipynb)**: Handoffs와 Router 패턴을 배웁니다.
